In [1]:
import uuid

from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from llm import model

In [2]:
class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]

def generate_topic(state: State):
    """LLM call to generate a topic for the joke"""
    msg = model.invoke("帮我生成一个笑话主题，只要返回笑话主题即可，不要返回额外其他任何信息。")
    return {"topic": msg.content}

def write_joke(state: State):
    """LLM call to write a joke based on the topic"""
    msg = model.invoke(f"请写一个关于{state['topic']}的笑话。只要返回笑话内容本身即可，不要返回任何其他信息。")
    return {"joke": msg.content}

In [6]:
config = {
    "configurable": {
        "thread_id": "111",
    }
}

# Build workflow
workflow = StateGraph(State)

# Add nodes
workflow.add_node("generate_topic", generate_topic)
workflow.add_node("write_joke", write_joke)

# Add edges to connect nodes
workflow.add_edge(START, "generate_topic")
workflow.add_edge("generate_topic", "write_joke")
workflow.add_edge("write_joke", END)

# Compile
checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

result = graph.invoke({}, config)
print(result)

{'topic': '程序员与咖啡机的误会', 'joke': '程序员新买了台智能咖啡机，兴奋地连上Wi-Fi，写了个脚本让它每天早上7点自动煮咖啡。  \n结果第二天醒来，发现厨房一片狼藉——咖啡机在凌晨3点开始疯狂煮咖啡，一直没停。  \n他检查代码，发现写的是“while (true) { brew(); }”，忘了加时间判断。  \n咖啡机委屈地说：“你让我无限循环，又怪我太肝？”'}


In [9]:
states = list(graph.get_state_history(config))

for state in states:
    print(state.next)
    print(state.config["configurable"]["checkpoint_id"])
    print()

()
1f142318-0d57-6482-8002-0ecc85672c43

('write_joke',)
1f142317-e4a6-6250-8001-93bfd742a064

('generate_topic',)
1f142317-dbd1-60aa-8000-786d34060f0b

('__start__',)
1f142317-dbcf-6cb0-bfff-69e7921df349



In [10]:
# 获取指定checkpointer
selected_state = states[1]
print(selected_state.next)
print(selected_state.values)
checkpointer_id = selected_state.config['configurable']['checkpoint_id']
print(checkpointer_id)

('write_joke',)
{'topic': '程序员与咖啡机的误会'}
1f142317-e4a6-6250-8001-93bfd742a064


In [11]:
# 从指定checkpointer执行
config_with_checkpoint = {
    "configurable": {
        "thread_id": config["configurable"]["thread_id"],
        "checkpoint_id": checkpointer_id
    }
}
# 初始值设置为None,表示不修改原state值重新执行
graph.invoke(None, config=config_with_checkpoint)

{'topic': '程序员与咖啡机的误会',
 'joke': '程序员新买了台智能咖啡机，说明书上写着：“请先连接Wi-Fi，再按‘开始’键。”\n\n他照做了，但咖啡机毫无反应。他检查代码、重启路由器、重装APP，折腾两小时后怒吼：“这破机器连个API文档都没有！”\n\n这时同事路过，瞥了一眼说：“你插电源了吗？”\n\n程序员愣住，低头一看——咖啡机根本没通电。\n\n他叹了口气：“原来是个硬件bug。”'}

In [12]:
# 我们如果对于State中的某些属性的值不满意，也可以修改后继续执行，但是修改状态后会产生一个新的checkpointer,不会对原来的checkpointer造成影响。
new_config = graph.update_state(selected_state.config, values={"topic": "小鸟"})

graph.invoke(None, config=new_config)

{'topic': '小鸟',
 'joke': '一只小鸟去理发店，  \n理发师问：“想要什么发型？”  \n小鸟说：“剪个秃头吧。”  \n理发师惊讶：“为什么？”  \n小鸟叹气：“最近掉毛太严重了，干脆一了百了！”'}